# IN inference pipeline

Runs the shared trained model (`outputs/models/{MODEL_NAME}/`, downloaded from S3 if missing) on IN counties: raw tiles -> canonical chips -> inference -> detections uploaded to `s3://.../IN/inference/{MODEL_NAME}/`.

Every state has its own copy of this notebook so per-state imagery handling can diverge; everything state-specific (paths, S3 prefixes, canonical CRS, county list, imagery source, county boundaries) comes from `src/utils/states.py` via `STATE`.

Use the **Python (vrt)** kernel (see `setup_env.ipynb`).

## Setup

In [ ]:
from pathlib import Path
import sys
import os

project_root = Path.cwd().resolve().parent  # notebooks -> root
sys.path.insert(0, str(project_root / "src"))

from utils import states

STATE = "IN"
MODEL_NAME = states.DEFAULT_MODEL

# Absolute project path on SageMaker, used in the printed nohup commands
REMOTE_PROJECT_ROOT = "/home/sagemaker-user/runoff_classifier"

state_cfg = states.get_state(STATE)
all_counties = list(state_cfg.counties)
county_data_dir = states.counties_dir(STATE)

print(f"State: {state_cfg.name} ({STATE}), FIPS {state_cfg.fips}, {len(all_counties)} counties")
print(f"Canonical CRS: {state_cfg.canonical_crs or 'NOT CONFIGURED'}")
print(f"Imagery source: {state_cfg.imagery_source or 'NOT IMPLEMENTED'}")
print(f"Model: {MODEL_NAME}")

## Inputs

Download the model (if not already local), the state's NHD waterway mask, the shared feature size thresholds, and the state's imagery-year pins (if any).

In [ ]:
import subprocess
from utils.model_store import ensure_model_local

model_path = ensure_model_local(MODEL_NAME)
print(f"Model: {model_path}")

# NHD mask (state-specific)
subprocess.check_call([
    "aws", "s3", "sync",
    states.s3_uri(states.s3_nhd_mask_prefix(STATE)) + "/",
    str(states.nhd_mask_dir(STATE)),
])
mask_path = states.nhd_mask_path(STATE)
if not mask_path.exists():
    print(f"WARNING: NHD mask not found at {mask_path} - upload nhd_mask_{STATE.lower()}.* "
          f"to {states.s3_uri(states.s3_nhd_mask_prefix(STATE))}/")

# Feature size thresholds (shared across states)
feat_size_thresh_path = states.feature_size_thresholds_path()
feat_size_thresh_path.parent.mkdir(parents=True, exist_ok=True)
subprocess.check_call([
    "aws", "s3", "cp",
    states.s3_uri(f"{states.s3_shared_prefix()}/feature_size_threshholds.csv"),
    str(feat_size_thresh_path),
])

# Imagery-year pins (optional; counties without a pin auto-detect the newest complete year)
years_csv = states.imagery_years_csv(STATE)
years_csv.parent.mkdir(parents=True, exist_ok=True)
rc = subprocess.call([
    "aws", "s3", "cp", states.s3_uri(states.s3_imagery_years_key(STATE)), str(years_csv),
])
print("imagery-year pins:", years_csv if rc == 0 else "none for this state")

print("downloads complete")

Select counties for inference

In [ ]:
import glob
from ipywidgets import SelectMultiple, Button, VBox
from IPython.display import display

counties_with_local_tiles = {
    c for c in all_counties
    if glob.glob(str(states.canonical_tiles_dir(STATE, c) / "*.tif"))
}
print(f"Counties with local canonical tiles: {len(counties_with_local_tiles)}")

cty_sel_infer = SelectMultiple(
    options=all_counties,
    value=(),
    description='counties:',
    rows=min(10, len(all_counties))
)

btn = Button(description="infer on these", button_style='info')

def confirm(_):
    selected = list(cty_sel_infer.value)
    print(f"selected counties ({len(selected)}): {selected}")
    print(f"  already local: {[c for c in selected if c in counties_with_local_tiles]}")

btn.on_click(confirm)
display(VBox([cty_sel_infer, btn]))

Verify / download canonical chips for the selected counties

Prints a `nohup` command to run chip preparation detached from the kernel (manifest fast path for counties already processed, full mosaic pipeline otherwise). Wait for its final `done` log line before running inference.

In [ ]:
import glob
import shlex

counties_to_infer = list(cty_sel_infer.value)

# ALWAYS re-verify every selected county - never skip based on a local-directory
# existence check: canonical_tiles/ is created (and its old chips cleared) EARLY in
# an ensure run, so an interrupted run can leave a partial or empty directory.
# prepare_canonical_mosaic.py is idempotent and self-healing (manifest fast path with
# size verification for completed counties), so re-running it IS the completeness check.

# Track which counties had no local chips before this run (for cleanup after inference)
infer_downloads_new = {
    c for c in counties_to_infer
    if not glob.glob(str(states.canonical_tiles_dir(STATE, c) / "*.tif"))
}

if counties_to_infer:
    counties_arg = shlex.quote(",".join(sorted(counties_to_infer)))
    log_dir = f"{REMOTE_PROJECT_ROOT}/outputs/{STATE}/logs"
    log_path = f"{log_dir}/canonical_mosaic_$(date +%Y%m%d_%H%M%S).log"
    # -u = unbuffered stdout, so the log doesn't look frozen while healthy
    cmd = (
        f"mkdir -p {log_dir} && "
        f"cd {REMOTE_PROJECT_ROOT} && "
        f"nohup python -u {REMOTE_PROJECT_ROOT}/scripts/prepare_canonical_mosaic.py "
        f"--state {STATE} --counties {counties_arg} --max_workers 16 "
        f"> {log_path} 2>&1 &"
    )

    print("Paste this into a terminal (not the notebook kernel) to verify/download detached:\n")
    print(cmd)
    print("\n# then tail the live log (adjust the filename to whatever timestamp was actually used):")
    print(f"tail -f {log_dir}/canonical_mosaic_*.log")
    print("\n# confirm it's still running:")
    print("ps aux | grep prepare_canonical_mosaic")
    print("\nWait for the final '[prepare_canonical_mosaic] done' line before running inference.")
else:
    print("No counties selected")

print(f"\nTiles to cleanup after inference: {infer_downloads_new if infer_downloads_new else 'none'}")

Setup inference parameters

In [ ]:
infer_config = dict(states.DEFAULT_INFER_CONFIG)  # shared with scripts/run_statewide_inference.py

print("Inference Configuration:")
for k, v in infer_config.items():
    print(f"  {k}: {v}")
print(f"\nModel: {model_path}")
print(f"Mask: {mask_path}")
print(f"Size thresholds: {feat_size_thresh_path}")

### Run inference

In [ ]:
# Setup environment
%env CUDA_VISIBLE_DEVICES=0,1,2,3
%env OMP_NUM_THREADS=4
%env MKL_NUM_THREADS=4
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

from datetime import datetime

from utils.county_boundaries import get_county_boundary_path

# Generate timestamp for output files
datestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Run inference on each selected county sequentially
counties_to_infer = list(cty_sel_infer.value)
print(f"Running inference on {len(counties_to_infer)} county(counties):")
for c in counties_to_infer:
    print(f"  - {c}")

for selected_county in counties_to_infer:
    print("\n" + "="*70)
    print(f"INFERENCE: {STATE} / {selected_county}")
    print("="*70)

    # Setup paths for this county
    infer_raster = str(states.canonical_tiles_dir(STATE, selected_county)) + os.sep
    infer_out_dir = states.county_inference_dir(STATE, selected_county, MODEL_NAME)
    os.makedirs(infer_out_dir, exist_ok=True)

    # Output filename with county and timestamp
    output_gpkg = infer_out_dir / f"detections_{selected_county}_{datestamp}.gpkg"

    # Resolve this county's TIGER/Line boundary polygon HERE (single process), before
    # launching the 4-rank torch.distributed command - so the one-time national TIGER
    # download / per-county GeoJSON extraction can't be raced by multiple ranks.
    # The state FIPS is required: many county names exist in several states.
    # Detections falling majority-outside this boundary are dropped by inference.py.
    county_boundary_path = get_county_boundary_path(selected_county, state_fips=state_cfg.fips)
    print(f"County boundary: {county_boundary_path}")

    # Build inference command
    cmd_infer = f"""python -m torch.distributed.run --nproc_per_node=4 \"{project_root / 'scripts' / 'inference.py'}\" \
        --task {infer_config['task']} \
        --device cuda \
        --raster_path \"{infer_raster}\" \
        --checkpoint \"{model_path}\" \
        --out_vector \"{output_gpkg}\" \
        --tile_size {infer_config['tile_size']} \
        --stride {infer_config['stride']} \
        --infer_batch {infer_config['infer_batch']} \
        --score_thresh {infer_config['score_thresh']} \
        --normalize {infer_config['normalize']} \
        --nms_iou_thresh {infer_config['nms_iou_thresh']} \
        --final_box_iou {infer_config['final_box_iou']} \
        --mask_path \"{mask_path}\" \
        --min_cover_frac {infer_config['min_cover_frac']} \
        --county_mask_path \"{county_boundary_path}\" \
        --county_min_cover_frac {infer_config['county_min_cover_frac']} \
        --class_area_csv \"{feat_size_thresh_path}\"\
    """

    print(f"County: {selected_county}")
    print(f"Input raster: {infer_raster}")
    print(f"Output: {output_gpkg}")
    print(f"Model: {model_path}\n")

    os.system(cmd_infer)

    print(f"Inference complete: {selected_county}")

print("\n" + "="*70)
print(f"Inference complete on all {len(counties_to_infer)} county(counties)")
print("="*70)

Clean up local tiles after inference (to save disk space)

Select counties to clean up. Counties with training labels (`*.shp`) in their folder are excluded so their chips stay available for training.

In [ ]:
import glob
from ipywidgets import SelectMultiple, Button, VBox
from IPython.display import display

counties_with_tiles_to_cleanup = sorted([
    c for c in all_counties
    if (states.raw_tiles_dir(STATE, c).is_dir() or states.canonical_tiles_dir(STATE, c).is_dir())
    and not glob.glob(str(states.county_dir(STATE, c) / "*.shp"))
])

print(f"Counties (without training labels) with local tiles available for cleanup: {len(counties_with_tiles_to_cleanup)}")

# Let user select which ones to clean up
cleanup_selector = SelectMultiple(
    options=counties_with_tiles_to_cleanup,
    value=tuple(counties_with_tiles_to_cleanup[:5]) if counties_with_tiles_to_cleanup else (),  # default: first 5
    description='cleanup:',
    rows=min(10, len(counties_with_tiles_to_cleanup))
)

cleanup_btn = Button(description="cleanup selected", button_style='warning')

def confirm_cleanup(_):
    selected_for_cleanup = list(cleanup_selector.value)
    print(f"Will cleanup {len(selected_for_cleanup)} county(counties): {selected_for_cleanup}")

cleanup_btn.on_click(confirm_cleanup)
display(VBox([cleanup_selector, cleanup_btn]))

Execute cleanup

In [ ]:
import os
import shutil

# Cleanup only the counties user selected
cleanup_count = 0
cleanup_size_gb = 0
counties_to_cleanup = list(cleanup_selector.value)

if not counties_to_cleanup:
    print("No counties selected for cleanup")
else:
    print(f"Cleaning up tiles for {len(counties_to_cleanup)} county(counties)...")

    for county in counties_to_cleanup:
        county_size_gb = 0.0
        found_any = False

        for subdir_name in ("tiles", "canonical_tiles"):
            subdir = os.path.join(states.county_dir(STATE, county), subdir_name)
            if not os.path.isdir(subdir):
                continue
            found_any = True

            size_bytes = sum(
                os.path.getsize(os.path.join(dirpath, filename))
                for dirpath, dirnames, filenames in os.walk(subdir)
                for filename in filenames
            )
            county_size_gb += size_bytes / (1024**3)

            shutil.rmtree(subdir)

        if found_any:
            cleanup_count += 1
            cleanup_size_gb += county_size_gb
            print(f"  Deleted {county}: {county_size_gb:.2f} GB")
        else:
            print(f"  {county}: no local tiles/canonical_tiles found")

    print(f"\nCleanup complete:")
    print(f"  Counties cleaned: {cleanup_count}")
    print(f"  Total space freed: {cleanup_size_gb:.2f} GB")
    print("\nCounties with training labels are never offered for cleanup.")

## Upload final county detections to S3

Uploads each county's final merged `detections_{county}_{timestamp}.gpkg` (per-rank partials excluded; latest timestamp wins if a county has several) from `outputs/{STATE}/inference/{MODEL_NAME}/{county}/` to `s3://.../{STATE}/inference/{MODEL_NAME}/`.

In [ ]:
import glob
import os

import boto3


def latest_final_detections(state, model):
    """{county: path} of the newest final merged detections gpkg per county."""
    latest = {}
    for path in sorted(glob.glob(str(states.inference_dir(state, model) / "*" / "detections_*.gpkg"))):
        name = os.path.basename(path)
        county = states.final_detections_county(name)
        if county is None:
            continue
        if county not in latest or name > os.path.basename(latest[county]):
            latest[county] = path
    return latest


latest_per_county = latest_final_detections(STATE, MODEL_NAME)
s3_prefix = states.s3_inference_prefix(STATE, MODEL_NAME)
s3_client = boto3.client("s3")

print(f"Found final detections for {len(latest_per_county)} county(ies) under {states.inference_dir(STATE, MODEL_NAME)}")
print(f"Uploading to {states.s3_uri(s3_prefix)}/\n")

uploaded, skipped = 0, 0
for county in sorted(latest_per_county):
    path = latest_per_county[county]
    key = f"{s3_prefix}/{os.path.basename(path)}"
    size = os.path.getsize(path)

    # skip if already on S3 with the same size
    try:
        head = s3_client.head_object(Bucket=states.S3_BUCKET, Key=key)
        if head["ContentLength"] == size:
            print(f"  skip (already uploaded): {os.path.basename(path)}")
            skipped += 1
            continue
    except s3_client.exceptions.ClientError:
        pass

    s3_client.upload_file(path, states.S3_BUCKET, key)
    print(f"  uploaded: {os.path.basename(path)} ({size / 1024**2:.1f} MB)")
    uploaded += 1

print(f"\nDone: {uploaded} uploaded, {skipped} skipped (already present)")

## Detection count summary CSV

Per-county breakdown of detection counts by class from each county's final merged detections gpkg (same file selection as the upload cell above), plus a per-county total column and a final STATEWIDE row. Saved to `outputs/{STATE}/inference/{MODEL_NAME}/detection_counts_summary.csv`.

In [ ]:
import geopandas as gpd
import pandas as pd

latest_per_county = latest_final_detections(STATE, MODEL_NAME)
print(f"Counting detections in {len(latest_per_county)} county gpkg(s)...")

rows = []
for county in sorted(latest_per_county):
    gdf = gpd.read_file(latest_per_county[county], ignore_geometry=True)
    counts = gdf["classname"].value_counts().to_dict()
    counts["County"] = county
    rows.append(counts)

summary = pd.DataFrame(rows).set_index("County").fillna(0).astype(int)
summary = summary[sorted(summary.columns)]  # stable class-column order
summary["Total"] = summary.sum(axis=1)

# Statewide totals row (per-class + grand total)
summary.loc["STATEWIDE"] = summary.sum(axis=0)

csv_path = states.inference_dir(STATE, MODEL_NAME) / "detection_counts_summary.csv"
summary.to_csv(csv_path)

print(f"\nSaved: {csv_path}\n")
print(summary.to_string())

### Statewide canonical tile coverage map

Reads the state's county canonical-chip manifests from S3 and plots their unique tile boundaries over all of the state's counties. No imagery is downloaded; the TIGER county boundary ZIP is downloaded only if needed.

Footprints are reconstructed from each manifest's canonical CRS, chip size, and filename row/column coordinates, assuming the pipeline's canonical six-inch output resolution (0.5 feet/pixel). The EPSG tag in a chip filename identifies source provenance, not the footprint CRS. Shared footprints are deduplicated. Missing county manifests are reported and outlined in red. These are manifest-listed footprints, not verified valid-pixel coverage.

In [ ]:
import json
import re

import boto3
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D
from shapely.geometry import box

from utils import county_boundaries as cb
from utils.common import normalize_county_key
from utils.prepare_reprojected_tiles import validate_manifest

map_counties = gpd.read_file(cb.ensure_national_county_zip())
map_counties = map_counties.loc[map_counties["STATEFP"] == state_cfg.fips].copy()
if len(map_counties) != len(state_cfg.counties) or map_counties.crs is None:
    raise ValueError(f"Expected {len(state_cfg.counties)} {STATE} county boundaries with a declared CRS.")
map_counties = map_counties.to_crs("EPSG:5070")
map_state_geom = map_counties.geometry.union_all()
map_state_keys = set(map_counties["NAME"].map(normalize_county_key))

map_s3 = boto3.client("s3")
map_prefix = states.s3_canonical_prefix(STATE) + "/manifests/"
map_manifest_keys = []
for page in map_s3.get_paginator("list_objects_v2").paginate(
    Bucket=states.S3_BUCKET, Prefix=map_prefix
):
    map_manifest_keys.extend(
        obj["Key"] for obj in page.get("Contents", [])
        if obj["Key"].endswith(".json")
    )

map_chip_pattern = re.compile(r"chip_\d+_r(-?\d+)_c(-?\d+)\.tif")
map_groups = {}
map_found_counties = set()
map_references = 0
for key in sorted(map_manifest_keys):
    response = map_s3.get_object(Bucket=states.S3_BUCKET, Key=key)
    manifest = json.loads(response["Body"].read())
    validate_manifest(STATE, manifest["county"], manifest)
    county_key = normalize_county_key(manifest["county"])
    if county_key not in map_state_keys:
        continue
    map_found_counties.add(county_key)
    crs = manifest["canonical_crs"]
    chip_px = manifest["chip_size_px"]
    if not isinstance(chip_px, int) or isinstance(chip_px, bool) or chip_px <= 0:
        raise ValueError(f"Invalid chip size in {key}: {chip_px!r}")
    chip_width = chip_px * states.CANONICAL_RES
    cells = map_groups.setdefault((crs, chip_width), set())
    if not manifest["keys"]:
        print(f"WARNING: empty chip manifest for {manifest['county']}: {key}")
    for chip_key in manifest["keys"]:
        match = map_chip_pattern.fullmatch(chip_key.rsplit("/", 1)[-1])
        if match is None:
            raise ValueError(f"Unrecognized chip filename in {key}: {chip_key}")
        cells.add(tuple(map(int, match.groups())))
        map_references += 1

map_missing = sorted(map_state_keys - map_found_counties)
print(f"{STATE} county manifests found: {len(map_found_counties)}/{len(map_state_keys)}")
print(f"Manifest chip references: {map_references:,}")
print(f"Grid reconstruction assumes canonical resolution = {states.CANONICAL_RES} feet/pixel.")
if map_missing:
    print("WARNING: counties without manifests:", ", ".join(map_missing))
if not map_references:
    raise RuntimeError(f"No {STATE} canonical chips found in manifests.")

map_frames = []
for (crs, width), cells in map_groups.items():
    if not cells:
        continue
    geometries = [
        box(col * width, row * width, (col + 1) * width, (row + 1) * width)
        for row, col in sorted(cells)
    ]
    map_frames.append(
        gpd.GeoDataFrame(geometry=geometries, crs=crs).to_crs(map_counties.crs)
    )
map_chips = gpd.GeoDataFrame(
    pd.concat(map_frames, ignore_index=True), crs=map_counties.crs
)
map_chips = map_chips.loc[~map_chips.geometry.to_wkb().duplicated()].copy()
map_chips = map_chips.loc[map_chips.intersects(map_state_geom)].copy()
if map_chips.empty:
    raise ValueError(f"No manifest chip footprints intersect {STATE}; check manifest CRS/grid metadata.")
print(f"Unique chip footprints intersecting {STATE}: {len(map_chips):,}")

fig, ax = plt.subplots(figsize=(14, 18))
map_chips.boundary.plot(ax=ax, color="darkorange", linewidth=0.25, alpha=0.65)
map_counties.boundary.plot(ax=ax, color="black", linewidth=0.6)
gpd.GeoSeries([map_state_geom], crs=map_counties.crs).boundary.plot(
    ax=ax, color="black", linewidth=1.5
)
if map_missing:
    map_counties.loc[
        map_counties["NAME"].map(normalize_county_key).isin(map_missing)
    ].boundary.plot(ax=ax, color="red", linewidth=1.2, linestyle="--")

xmin, ymin, xmax, ymax = map_counties.total_bounds
ax.set_xlim(xmin - 5000, xmax + 5000)
ax.set_ylim(ymin - 5000, ymax + 5000)
ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title(
    f"{state_cfg.name} canonical six-inch chip footprints\n"
    f"{len(map_chips):,} unique footprints; "
    f"{len(map_found_counties)}/{len(map_state_keys)} county manifests"
)
map_handles = [
    Line2D([0], [0], color="darkorange", label="Canonical chip boundaries"),
    Line2D([0], [0], color="black", label="County boundaries"),
]
if map_missing:
    map_handles.append(
        Line2D([0], [0], color="red", linestyle="--", label="Missing manifest")
    )
ax.legend(handles=map_handles, loc="upper right")
plt.show()